# Primitives

## What Is a Primitive?

In [1]:
import jax
import jax.numpy as jnp
import jax.lax as lax

In [2]:
print(f"sin primitive: {lax.sin_p}")
print(f"Type: {type(lax.sin_p).__name__}")

sin primitive: sin
Type: Primitive


In [3]:
for name in ["add_p", "mul_p", "sin_p", "cos_p", "exp_p"]:
    if hasattr(lax, name):
        print(f"    {name}")

    add_p
    mul_p
    sin_p
    cos_p
    exp_p


## Tracing Through a Primitive

In [4]:
def f(x):
    return jnp.sin(x)


print(jax.make_jaxpr(f)(jnp.array(1.0)))

{ lambda ; a:f32[]. let b:f32[] = sin a in (b,) }


In [5]:
x = jnp.array(1.0)

print("=== StableHLO for sin ===")
print(jax.jit(f).lower(x).as_text())

=== StableHLO for sin ===
module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<f32>) -> (tensor<f32> {jax.result_info = "result"}) {
    %0 = stablehlo.sine %arg0 : tensor<f32>
    return %0 : tensor<f32>
  }
}



In [6]:
print("=== Gradient of sin ===")
print(jax.make_jaxpr(jax.grad(f))(x))
print()
print(f"sin'(x) = cos(x) = {jax.grad(f)(x)}")
print(f"cos(x) = {jnp.cos(x)}")

=== Gradient of sin ===
{ lambda ; a:f32[]. let
    _:f32[] = sin a
    b:f32[] = cos a
    c:f32[] = mul 1.0:f32[] b
  in (c,) }

sin'(x) = cos(x) = 0.5403022766113281
cos(x) = 0.5403022766113281


## Implementing a Custom Primitive

### Step 1: Definition Implementation

In [7]:
from jax._src import core
import numpy as np

In [8]:
# Create the primitive
xlogy_p = core.Primitive("xlogy")
xlogy_p.multiple_results = False

In [9]:
@xlogy_p.def_impl
def xlogy_impl(x, y):
    return np.where(x == 0.0, 0.0, x * np.log(y))

In [10]:
result = xlogy_p.bind(np.array([0.0, 1.0, 2.0]), np.array([1.0, 2.0, 3.0]))
print(f"Impl result: {result}")

Impl result: [0.        0.6931472 2.1972246]


### Step 2: Abstract Evaluation

In [11]:
from jax.core import ShapedArray


@xlogy_p.def_abstract_eval
def xlogy_abstract(x_aval, _):
    print(f"x_aval: {x_aval}")
    return ShapedArray(x_aval.shape, x_aval.dtype)

In [12]:
x = jnp.array([0.0, 1.0, 2.0])
y = jnp.array([1.0, 2.0, 3.0])

In [13]:
def f(x, y):
    return xlogy_p.bind(x, y)

In [14]:
print("Jaxpr with our primitive")
print(jax.make_jaxpr(f)(x, y))

Jaxpr with our primitive
x_aval: float32[3]
{ lambda ; a:f32[3] b:f32[3]. let c:f32[3] = xlogy a b in (c,) }


### Step 3: JIT Lowering

In [15]:
from jax._src.interpreters import mlir as jax_mlir
from jax._src.lib.mlir.dialects import hlo


def xlogy_lowering(ctx, x, y):
    log_y = hlo.LogOp(y).result
    prod = hlo.MulOp(x, log_y).result

    aval = ctx.avals_in[0]
    zero = jax_mlir.full_like_aval(ctx, 0.0, aval)

    is_zero = hlo.CompareOp(
        x,
        zero,
        hlo.ComparisonDirectionAttr.get("EQ"),
        compare_type=hlo.ComparisonTypeAttr.get("FLOAT"),
    ).result

    return [hlo.SelectOp(is_zero, zero, prod).result]

In [16]:
jax_mlir.register_lowering(xlogy_p, xlogy_lowering)

In [17]:
jax.jit(f)(x, y)

Array([0.       , 0.6931472, 2.1972246], dtype=float32)

In [18]:
print(jax.jit(f).lower(x, y).as_text())

module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<3xf32>, %arg1: tensor<3xf32>) -> (tensor<3xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.log %arg1 : tensor<3xf32>
    %1 = stablehlo.multiply %arg0, %0 : tensor<3xf32>
    %cst = stablehlo.constant dense<0.000000e+00> : tensor<f32>
    %2 = stablehlo.broadcast_in_dim %cst, dims = [] : (tensor<f32>) -> tensor<3xf32>
    %3 = stablehlo.compare EQ, %arg0, %2, FLOAT : (tensor<3xf32>, tensor<3xf32>) -> tensor<3xi1>
    %4 = stablehlo.select %3, %2, %1 : tensor<3xi1>, tensor<3xf32>
    return %4 : tensor<3xf32>
  }
}



### JVP Rule

In [19]:
from jax.interpreters import ad
from jax._src.interpreters.ad import Zero


def xlogy_jvp(primals, tangents):
    x, y = primals
    x_dot, y_dot = tangents

    primal_out = xlogy_p.bind(x, y)

    t_x = jnp.zeros_like(x) if isinstance(x_dot, Zero) else jnp.log(y) * x_dot
    t_y = jnp.zeros_like(y) if isinstance(y_dot, Zero) else (x / y) * y_dot
    tangent_out = t_x + t_y

    return primal_out, tangent_out

In [20]:
ad.primitive_jvps[xlogy_p] = xlogy_jvp

In [21]:
def f(x, y):
    return jnp.sum(xlogy_p.bind(x, y))


grad_x, grad_y = jax.grad(f, argnums=(0, 1))(x, y)
print(f"Grad w.r.t. x: {grad_x}")
print(f"log(y): {jnp.log(y)}")
print()
print(f"Grad w.r.t. y: {grad_y}")
print(f"x/y: {x/y}")

Grad w.r.t. x: [0.        0.6931472 1.0986123]
log(y): [0.        0.6931472 1.0986123]

Grad w.r.t. y: [0.        0.5       0.6666667]
x/y: [0.        0.5       0.6666667]


## The Practical Path: `custom_vjp`

In [22]:
@jax.custom_vjp
def stable_log1p(x):
    return jnp.log1p(x)

In [23]:
def stable_log1p_fwd(x):
    return stable_log1p(x), x

In [24]:
def stable_log1p_bwd(residuals, g):
    x = residuals
    return (g / (1.0 + x),)

In [25]:
stable_log1p.defvjp(stable_log1p_fwd, stable_log1p_bwd)

In [26]:
x = jnp.array([0.1, 0.5, 1.0, 2.0])
print("stable_log1p(x):", stable_log1p(x))
print("grad:", jax.grad(lambda x: jnp.sum(stable_log1p(x)))(x))
print("expected:", 1.0 / (1.0 + x))

stable_log1p(x): [0.09531018 0.4054651  0.6931472  1.0986123 ]
grad: [0.9090909  0.6666667  0.5        0.33333334]
expected: [0.9090909  0.6666667  0.5        0.33333334]


## `custom_vjp` for a Numerically Stable Sigmoid Gradient

In [27]:
@jax.custom_vjp
def stable_sigmoid(x):
    return 1.0 / (1.0 + jnp.exp(-x))

In [28]:
def stable_sigmoid_fwd(x):
    y = stable_sigmoid(x)
    return y, y

In [29]:
def stable_sigmoid_bwd(y, g):
    return (g * y * (1.0 - y),)

In [30]:
stable_sigmoid.defvjp(stable_sigmoid_fwd, stable_sigmoid_bwd)

In [31]:
def naive_sigmoid(x):
    return 1.0 / (1.0 + jnp.exp(-x))


x = jnp.array([-100.0, -10.0, 0.0, 10.0, 100.0])

In [32]:
print(f"forward: {stable_sigmoid(x)}")
print("naive autodiff grad:", jax.grad(lambda x: jnp.sum(naive_sigmoid(x)))(x))
print("custom_vjp grad:", jax.grad(lambda x: jnp.sum(stable_sigmoid(x)))(x))

forward: [0.0000000e+00 4.5397872e-05 5.0000000e-01 9.9995458e-01 1.0000000e+00]
naive autodiff grad: [          nan 4.5395813e-05 2.5000000e-01 4.5395806e-05 0.0000000e+00]
custom_vjp grad: [0.0000000e+00 4.5395809e-05 2.5000000e-01 4.5416677e-05 0.0000000e+00]


In [33]:
jax.nn.sigmoid

<PjitFunction of <function sigmoid at 0x0000016F7BD5EA20>>

## `custom_jvp` for Forward-Mode Custom Derivatives

In [ ]:
@jax.custom_jvp
def f(x):
    return jnp.sin(x) * jnp.cos(x)  # f(x) = sin(2x)/2 -- there was a mistake in the recording (was written as /x)

In [ ]:
@f.defjvp
def f_jvp(primals, tangents):
    (x,) = primals
    (x_dot,) = tangents
    # f'(x) = cos(2x) = cos^2(x) - sin^2(x)  -- there was a mistake in the recording (missing exponent on sin)
    primal_out = f(x)
    tangent_out = (jnp.cos(x) ** 2 - jnp.sin(x) ** 2) * x_dot
    return primal_out, tangent_out

In [36]:
x = jnp.array(1.0)
print(f"f(x): {f(x)}")
print(f"f'(x) via custom_jvp: {jax.grad(f)(x)}")
print(f"Expected (cos(2(x)): {jnp.cos(2*x)}")

f(x): 0.4546486735343933
f'(x) via custom_jvp: -0.41614681482315063
Expected (cos(2(x)): -0.416146844625473


## All Transformations Work on Custom Primitives

### batching rule

In [40]:
from jax.interpreters import batching

def xlogy_batch(batched_args, batch_dims):
    x, y = batched_args
    return xlogy_p.bind(x, y), batch_dims[0]

batching.primitive_batchers[xlogy_p] = xlogy_batch

In [41]:
x = jnp.array([0.0, 1.0, 2.0])
y = jnp.array([1.0, 2.0, 3.0])

In [44]:
# JIT
print(f"jit: {jax.jit(lambda x, y: xlogy_p.bind(x,y))(x,y)}")

# vmap
print(f"vmap: {jax.vmap(lambda x, y: xlogy_p.bind(x,y))(x,y)}")

# grad
print(f"grad: {jax.grad(lambda x: jnp.sum(xlogy_p.bind(x,y)))(x)}")

# jit(grad)
print(f"jit(grad): {jax.jit(jax.grad(lambda x: jnp.sum(xlogy_p.bind(x,y))))(x)}")

jit: [0.        0.6931472 2.1972246]
vmap: [0.        0.6931472 2.1972246]
grad: [0.        0.6931472 1.0986123]
jit(grad): [0.        0.6931472 1.0986123]
